In [85]:
from urllib.request import urlretrieve
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import zipfile
import bs4
import os
import re

In [57]:
SRC_LIST_PATH = os.path.join("data", "docs", "databank-list.txt")

with open(SRC_LIST_PATH, 'r') as f:
    databank_list = [src.strip('\n') for src in f.readlines()]

databank_list

['Agricultural land (% of land area)',
 'Food production index (2014-2016 = 100)',
 'Forest area (% of land area)',
 'Rural population (% of total population)',
 'Net migration',
 'Poverty headcount ratio at $3.00 a day (2021 PPP) (% of population)',
 'Pregnant women receiving prenatal care (%)',
 'Prevalence of HIV, total (% of population ages 15-49)',
 'Primary completion rate, total (% of relevant age group)',
 'School enrollment, primary and secondary (gross), gender parity index (GPI)',
 'Access to electricity (% of population)',
 'Population growth (annual %)',
 'Population in urban agglomerations of more than 1 million (% of total population)',
 'Population living in areas where elevation is below 5 meters (% of total population)',
 'Terrestrial and marine protected areas (% of total territorial area)',
 'Urban population (% of total population)',
 'Central government debt, total (% of GDP)',
 'Expense (% of GDP)',
 'GDP growth (annual %)',
 'GDP per capita growth (annual %)',
 

In [56]:
DATABANK_URL = "https://data.worldbank.org/indicator"
DATABANK_INDICATORS_PATH = os.path.join("data", "html", "databank-indicators.html")

urlretrieve(DATABANK_URL, DATABANK_INDICATORS_PATH)

('data\\html\\databank-indicators.html',
 <http.client.HTTPMessage at 0x1b98a3d91d0>)

In [40]:
with open(DATABANK_INDICATORS_PATH, 'r', encoding='utf-8') as f:
    html_text = f.read()
    soup = bs4.BeautifulSoup(html_text, 'html.parser')
    indicator_links = soup.find_all('a', href=re.compile("/indicator/"))

In [54]:
API_DOWNLOAD_URL = "https://api.worldbank.org/v2/en/indicator/"
download_links = []
for indicator_name in databank_list:
    name = indicator_name.replace(" ", "").lower()
    for link in indicator_links:
        if link.text.replace(" ", "").lower() == name:
            code = link.get("href").split("/")[2]
            download_url = f"{API_DOWNLOAD_URL}{code}?downloadformat=csv"
            download_links.append((indicator_name, code, download_url))
download_links

[('Agricultural land (% of land area)',
  'AG.LND.AGRI.ZS',
  'https://api.worldbank.org/v2/en/indicator/AG.LND.AGRI.ZS?downloadformat=csv'),
 ('Agricultural land (% of land area)',
  'AG.LND.AGRI.ZS',
  'https://api.worldbank.org/v2/en/indicator/AG.LND.AGRI.ZS?downloadformat=csv'),
 ('Agricultural land (% of land area)',
  'AG.LND.AGRI.ZS',
  'https://api.worldbank.org/v2/en/indicator/AG.LND.AGRI.ZS?downloadformat=csv'),
 ('Food production index (2014-2016 = 100)',
  'AG.PRD.FOOD.XD',
  'https://api.worldbank.org/v2/en/indicator/AG.PRD.FOOD.XD?downloadformat=csv'),
 ('Forest area (% of land area)',
  'AG.LND.FRST.ZS',
  'https://api.worldbank.org/v2/en/indicator/AG.LND.FRST.ZS?downloadformat=csv'),
 ('Forest area (% of land area)',
  'AG.LND.FRST.ZS',
  'https://api.worldbank.org/v2/en/indicator/AG.LND.FRST.ZS?downloadformat=csv'),
 ('Forest area (% of land area)',
  'AG.LND.FRST.ZS',
  'https://api.worldbank.org/v2/en/indicator/AG.LND.FRST.ZS?downloadformat=csv'),
 ('Rural population

In [84]:
for name, code, url in download_links:
    urlretrieve(url, os.path.join("data", "zip", f"{name}.zip"))

In [90]:
ZIP_PATH = os.path.join("data", "zip")
CSV_PATH = os.path.join("data", "csv")
for file in os.listdir(ZIP_PATH):
    file_dir = os.path.join(CSV_PATH, file.replace(".zip", ""))
    os.mkdir(file_dir)
    zipfile.ZipFile(os.path.join(ZIP_PATH, file), 'r').extractall(file_dir)

In [61]:
# Get happiness data
HAPPINESS_URL = "https://files.worldhappiness.report/WHR26_Data_Figure_2.1.xlsx"

# Automated retrieval is forbidden (403), so manual retrieval was needed.
# File is stored, manually, in defined location at `data/xlsx/happiness_report.xlsx`
# urlretrieve(HAPPINESS_URL, os.path.join("data", "xlsx", "happiness_report.xlsx"))

# Data preliminary exploration

## Happiness data

In [63]:
happiness = pd.read_excel(os.path.join("data", "xlsx", "happiness_report.xlsx"))
happiness.head()

,Year,Rank,Country name,Life evaluation (3-year average),Lower whisker,Upper whisker,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual
0,2025,1,Finland,7.764,7.690,7.837,1.915,1.638,0.939,1.105,0.093,0.491,1.582
1,2025,2,Iceland,7.540,7.449,7.630,1.971,1.720,0.996,1.105,0.187,0.187,1.373
2,2025,3,Denmark,7.539,7.446,7.631,1.986,1.633,0.930,1.081,0.125,0.474,1.310
3,2025,4,Costa Rica,7.439,7.356,7.522,1.697,1.483,0.739,1.101,0.059,0.122,2.236
4,2025,5,Sweden,7.255,7.172,7.337,1.950,1.570,1.027,1.070,0.149,0.447,1.041


In [64]:
happiness.info()

<class 'pandas.DataFrame'>
RangeIndex: 2116 entries, 0 to 2115
Data columns (total 13 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Year                                        2116 non-null   int64  
 1   Rank                                        2116 non-null   int64  
 2   Country name                                2116 non-null   str    
 3   Life evaluation (3-year average)            2116 non-null   float64
 4   Lower whisker                               1022 non-null   float64
 5   Upper whisker                               1022 non-null   float64
 6   Explained by: Log GDP per capita            1019 non-null   float64
 7   Explained by: Social support                1019 non-null   float64
 8   Explained by: Healthy life expectancy       1016 non-null   float64
 9   Explained by: Freedom to make life choices  1017 non-null   float64
 10  Explained by: Generosit

We are interested in the year, country name, and life evaluation average.

In [66]:
happiness["Life evaluation (3-year average)"].describe()

count    2116.000000
mean        5.465655
std         1.123870
min         1.364000
25%         4.604750
50%         5.480000
75%         6.321250
max         7.856000
Name: Life evaluation (3-year average), dtype: float64

In [73]:
happiness_df = happiness.loc[:, ["Year", "Country name", "Life evaluation (3-year average)"]]
MAX_YEAR, MIN_YEAR = happiness_df["Year"].max(), happiness_df["Year"].min()
COUNTRIES_LIST = happiness_df["Country name"].unique()

MAX_YEAR, MIN_YEAR, COUNTRIES_LIST

(np.int64(2025),
 np.int64(2011),
 <StringArray>
 [          'Finland',           'Iceland',           'Denmark',
         'Costa Rica',            'Sweden',            'Norway',
        'Netherlands',            'Israel',        'Luxembourg',
        'Switzerland',
  ...
             'Bhutan',             'Syria',             'Sudan',
             'Angola',       'Puerto Rico',          'Suriname',
  'Somaliland Region',          'Djibouti',            'Guyana',
               'Cuba']
 Length: 168, dtype: str)

## Bank data

In [93]:
indicator_names = os.listdir(CSV_PATH)
indicator_files = dict()
for indicator in indicator_names:
    files = os.listdir(os.path.join(CSV_PATH, indicator))
    for f in files:
        if not f.startswith("Metadata"):
            indicator_files[indicator] = os.path.join(CSV_PATH, indicator, f)
indicator_files

{'Access to electricity (% of population)': 'data\\csv\\Access to electricity (% of population)\\API_EG.ELC.ACCS.ZS_DS2_en_csv_v2_3606.csv',
 'Agricultural land (% of land area)': 'data\\csv\\Agricultural land (% of land area)\\API_AG.LND.AGRI.ZS_DS2_en_csv_v2_7781.csv',
 'Central government debt, total (% of GDP)': 'data\\csv\\Central government debt, total (% of GDP)\\API_GC.DOD.TOTL.GD.ZS_DS2_en_csv_v2_5449.csv',
 'Expense (% of GDP)': 'data\\csv\\Expense (% of GDP)\\API_GC.XPN.TOTL.GD.ZS_DS2_en_csv_v2_2867.csv',
 'Food production index (2014-2016 = 100)': 'data\\csv\\Food production index (2014-2016 = 100)\\API_AG.PRD.FOOD.XD_DS2_en_csv_v2_15376.csv',
 'Forest area (% of land area)': 'data\\csv\\Forest area (% of land area)\\API_AG.LND.FRST.ZS_DS2_en_csv_v2_4195.csv',
 'Fossil fuel energy consumption (% of total)': 'data\\csv\\Fossil fuel energy consumption (% of total)\\API_EG.USE.COMM.FO.ZS_DS2_en_csv_v2_7098.csv',
 'GDP growth (annual %)': 'data\\csv\\GDP growth (annual %)\\API_

**Note:** Every csv file starts at row 2!

### Access to electricity

In [97]:
electricity = pd.read_csv(indicator_files["Access to electricity (% of population)"], header=2)
electricity.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,100.000000,100.000000,100.000000,100.000000,100.000000,99.900000,100.000000,NaN,NaN,NaN
1,Africa Eastern and Southern,AFE,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,40.223744,43.035073,44.390861,46.282371,48.127211,48.801258,50.667516,NaN,NaN,NaN
2,Afghanistan,AFG,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,97.700000,93.400000,97.700000,97.700000,97.700000,85.300000,85.300000,NaN,NaN,NaN
3,Africa Western and Central,AFW,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,48.902228,51.338199,51.289374,51.851866,54.366012,55.683570,57.064737,NaN,NaN,NaN
4,Angola,AGO,Access to electricity (% of population),EG.ELC.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,42.900000,45.300000,45.600000,47.000000,48.200000,48.500000,51.100000,NaN,NaN,NaN


In [98]:
electricity.describe(), electricity.info()

<class 'pandas.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 71 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Country Name    265 non-null    str    
 1   Country Code    265 non-null    str    
 2   Indicator Name  265 non-null    str    
 3   Indicator Code  265 non-null    str    
 4   1960            0 non-null      float64
 5   1961            0 non-null      float64
 6   1962            0 non-null      float64
 7   1963            0 non-null      float64
 8   1964            0 non-null      float64
 9   1965            0 non-null      float64
 10  1966            0 non-null      float64
 11  1967            0 non-null      float64
 12  1968            0 non-null      float64
 13  1969            0 non-null      float64
 14  1970            0 non-null      float64
 15  1971            0 non-null      float64
 16  1972            0 non-null      float64
 17  1973            0 non-null      float64
 18  1

(       1960  1961  1962  1963  1964  1965  1966  1967  1968  1969  ...  \
 count   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
 mean    NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
 std     NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
 min     NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
 25%     NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
 50%     NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
 75%     NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
 max     NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN  ...   
 
              2017        2018        2019        2020        2021        2022  \
 count  262.000000  262.000000  262.000000  262.000000  262.000000  262.000000   
 mean    84.677662   85.315478   85.874072   86.408904   86.969251   87.263447   
 std     24.907559   24.049449   23.793819   23.387846   22.846938   22.45256

# Data preparation

In [156]:
# Melting the bank data df
id_vars = ["Country Name", "Indicator Name"]
year_cols = [col for col in electricity.columns if col.isdigit()]
electricity_long = pd.melt(electricity,
        id_vars=id_vars,
        value_vars=year_cols,
        var_name="Year",
        value_name="Value")

# For compatibility, transform year strings into integers:
electricity_long["Year"] = electricity_long["Year"].astype("int")

# Now, Indicator Name must be removed and the Value column should be renamed:
indicator_name = electricity_long["Indicator Name"][0]
electricity_long.rename(columns={"Value": indicator_name}, inplace=True)
electricity_long.drop(columns=["Indicator Name"], inplace=True)

# Also, the country column should be renamed for compatibility:
electricity_long.rename(columns={"Country Name":"Country name"}, inplace=True)

# Check the results
electricity_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 17490 entries, 0 to 17489
Data columns (total 3 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Country name                             17490 non-null  str    
 1   Year                                     17490 non-null  int64  
 2   Access to electricity (% of population)  7835 non-null   float64
dtypes: float64(1), int64(1), str(1)
memory usage: 410.1 KB


In [157]:
electricity_long.sample(5)

,Country name,Year,Access to electricity (% of population)
1300,Sub-Saharan Africa (IDA & IBRD countries),1964,NaN
11106,Trinidad and Tobago,2001,96.3
10498,Northern Mariana Islands,1999,100.0
13899,Kazakhstan,2012,100.0
14289,Tuvalu,2013,97.6


Note that some countries have different naming conventions, such as Yemen:
In "Happiness", it's called "Yemen", whereas in Bank data, it can be found as "Yemen, Rep.".

A naive approach will be taken, where we only replace the string ", Rep." and assume the rest of the countries are correctly formatted as the Happiness data's standard.

In [158]:
electricity_long.tail(5)

,Country name,Year,Access to electricity (% of population)
17485,Kosovo,2025,NaN
17486,"Yemen, Rep.",2025,NaN
17487,South Africa,2025,NaN
17488,Zambia,2025,NaN
17489,Zimbabwe,2025,NaN


In [159]:
electricity_long["Country name"] = electricity_long.loc[:, "Country name"].str.replace(", Rep.", "")
electricity_long.tail(5)

,Country name,Year,Access to electricity (% of population)
17485,Kosovo,2025,NaN
17486,Yemen,2025,NaN
17487,South Africa,2025,NaN
17488,Zambia,2025,NaN
17489,Zimbabwe,2025,NaN


In [160]:
# Reindex
pd.merge(happiness, electricity_long, on=["Country name", "Year"])

,Year,Rank,Country name,Life evaluation (3-year average),Lower whisker,Upper whisker,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,Explained by: Generosity,Explained by: Perceptions of corruption,Dystopia + residual,Access to electricity (% of population)
0,2025,1,Finland,7.764,7.690,7.837,1.915,1.638,0.939,1.105,0.093,0.491,1.582,NaN
1,2025,2,Iceland,7.540,7.449,7.630,1.971,1.720,0.996,1.105,0.187,0.187,1.373,NaN
2,2025,3,Denmark,7.539,7.446,7.631,1.986,1.633,0.930,1.081,0.125,0.474,1.310,NaN
3,2025,4,Costa Rica,7.439,7.356,7.522,1.697,1.483,0.739,1.101,0.059,0.122,2.236,NaN
4,2025,5,Sweden,7.255,7.172,7.337,1.950,1.570,1.027,1.070,0.149,0.447,1.041,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1891,2011,152,Burundi,3.678,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.2
1892,2011,153,Sierra Leone,3.586,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14.2
1893,2011,154,Central African Republic,3.568,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10.4
1894,2011,155,Benin,3.493,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36.9


## Data reunification
Now that our simple pipeline worked, we can iterate over every dataset to merge it with our happiness df.

In [163]:
def unify(indicator_name:str, final_df: pd.DataFrame, indicator_df: pd.DataFrame) -> pd.DataFrame:
    # Melting the bank data df
    id_vars = ["Country Name", "Indicator Name"]
    year_cols = [col for col in indicator_df.columns if col.isdigit()]
    df_melted = pd.melt(indicator_df,
            id_vars=id_vars,
            value_vars=year_cols,
            var_name="Year",
            value_name="Value")

    # For compatibility, transform year strings into integers:
    df_melted["Year"] = df_melted["Year"].astype("int")

    # Now, Indicator Name must be removed and the Value column should be renamed:
    df_melted.rename(columns={"Value": indicator_name}, inplace=True)
    df_melted.drop(columns=["Indicator Name"], inplace=True)

    # Also, the country column should be renamed for compatibility:
    df_melted.rename(columns={"Country Name": "Country name"}, inplace=True)

    # Naive correction of country names
    df_melted["Country name"] = df_melted.loc[:, "Country name"].str.replace(", Rep.", "")


    return pd.merge(final_df, df_melted, on=["Country name", "Year"])

In [164]:
final_df = happiness
for indicator_name, filepath in indicator_files.items():
    # Remember that files start at the second row!
    indicator_df = pd.read_csv(filepath, header=2)
    # Iteratively unify datasets
    final_df = unify(indicator_name, final_df, indicator_df)

In [165]:
final_df.head()

,Year,Rank,Country name,Life evaluation (3-year average),Lower whisker,Upper whisker,Explained by: Log GDP per capita,Explained by: Social support,Explained by: Healthy life expectancy,Explained by: Freedom to make life choices,...,Forest area (% of land area),Fossil fuel energy consumption (% of total),GDP growth (annual %),GDP per capita growth (annual %),"Inflation, consumer prices (annual %)","Life expectancy at birth, total (years)",Population growth (annual %),Pregnant women receiving prenatal care (%),Rural population (% of total population),Urban population (% of total population)
0,2025,1,Finland,7.764,7.690,7.837,1.915,1.638,0.939,1.105,...,NaN,NaN,0.173445,-0.297136,0.337875,NaN,0.470872,NaN,25.475216,74.524784
1,2025,2,Iceland,7.540,7.449,7.630,1.971,1.720,0.996,1.105,...,NaN,NaN,1.279462,-0.242812,4.090851,NaN,1.514453,NaN,5.782068,94.217932
2,2025,3,Denmark,7.539,7.446,7.631,1.986,1.633,0.930,1.081,...,NaN,NaN,2.928433,2.377287,1.893674,NaN,0.536904,NaN,11.154156,88.845844
3,2025,4,Costa Rica,7.439,7.356,7.522,1.697,1.483,0.739,1.101,...,NaN,NaN,4.560335,4.092823,-0.073671,NaN,0.448125,NaN,20.273186,79.726814
4,2025,5,Sweden,7.255,7.172,7.337,1.950,1.570,1.027,1.070,...,NaN,NaN,1.543164,1.285286,0.679869,NaN,0.254281,NaN,10.922132,89.077868


In [166]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1896 entries, 0 to 1895
Data columns (total 28 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Year                                         1896 non-null   int64  
 1   Rank                                         1896 non-null   int64  
 2   Country name                                 1896 non-null   str    
 3   Life evaluation (3-year average)             1896 non-null   float64
 4   Lower whisker                                919 non-null    float64
 5   Upper whisker                                919 non-null    float64
 6   Explained by: Log GDP per capita             917 non-null    float64
 7   Explained by: Social support                 917 non-null    float64
 8   Explained by: Healthy life expectancy        917 non-null    float64
 9   Explained by: Freedom to make life choices   915 non-null    float64
 10  Explained b

In [ ]:
def save_csv(filename: str, df: pd.DataFrame):
    path = os.path.join("data", "csv", filename + ".csv")
    df.to_csv(path)

save_csv("joint_data_v1", final_df)

## Data cleaning
As seen above, some features do not have enough data to be useful.